# Day 65：毕业答辩日：15 分钟高分答辩结构、企业评委追问清单与防翻车演示验收脚本（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 10 周**：毕业答辩、简历作品集与面试冲刺结业  
> **GitLab 工单**：`STARMIND-065` ｜ **开发分支**：`feat/day65-defense-demo-kit`  
> **配套 HTML5 课件**：`courseware/day65/index.html` ｜ **独立版本快照**：`versions/day65/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天站上毕业答辩台！老陈当过无数次大厂晋升评委和校招/社招终面官，见过太多技术很强但答辩吃亏的同学：前 10 分钟全在念背景和安装步骤，真正最值钱的架构权衡、压测数据和现场 Demo 只剩 1 分钟草草收尾，甚至现场网一断 Demo 直接社死！今天咱们用工程化思维解决答辩：第一，用代码校验 15 分钟黄金演讲节奏；第二，把企业评委最爱连环追问的 6 大难题提前演练透；第三，写一个带离线确定性降级快照的现场演示验收脚本，确保台上零翻车！

### 承上启下
- **昨日基线**：昨天我们完成了星辰智脑的容器化部署与交付手册，今天进入第 10 周就业冲刺第一战——企业级毕业答辩！
- **明日预告**：明天（Day 66）进入简历与作品集打磨：用 STAR 法则把星辰智脑包装成简历王牌项目，并自动生成高颜值 GitHub 主页 README。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day65'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-065）

学员需在 15 分钟内向由架构师、产品经理与技术总监组成的评审团完整汇报《星辰智脑》项目，并完成 5 分钟高频技术追问与现场功能演示。

**今日迭代目标：**
- 实现 15 分钟高分答辩结构规划与时间预算校验器（5 段黄金结构 + 评委评分维度模拟）。
- 实现可复现的现场演示脚本编排器：支持主链路实时运行 + 异常自动切换预录确定性快照，并生成答辩就绪度报告。

**验收标准：**
- 答辩规划器严格校验总时长为 15 分钟且核心架构+演示+指标占比 >= 65%。
- 演示验收器覆盖知识问答溯源、多 Agent 高危 HITL 审批、Ragas 指标展示三个核心场景，且在主服务异常时无缝降级。
- 纯 Python 离线运行，pytest 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 评委最容易给高分的三个细节

1) 敢于主动展示一个你解决过的真实 Bad Case（优化前错误 -> 排查定位 -> 优化后命中）；2) 清楚说出每个组件的成本（算清 1 万次问答花多少钱）；3) 架构图层次分明，不堆砌自己答不上来的中间件名词。

> **⚠️ 老陈避坑**：千万不要把没用过的技术（如 K8s 联邦集群、Milvus 百亿分片）硬塞进 PPT，评委顺手一问就会露馅。

### 2. 回答'如果流量暴涨 10 倍怎么扩容'的标准思路

分三层拆解：接入层（Nginx + 多副本无状态 FastAPI 水平扩容）、缓存与检索层（Redis 语义缓存 + 向量库读写分离）、推理层（vLLM 多卡 Tensor Parallelism + 分级路由 + 队列限流降级）。

> **⚠️ 老陈避坑**：不要笼统答'加机器'，要指出哪一层最先成为瓶颈（AI 应用通常是推理层 GPU 显存/算力与外部 API RPM 限流）。


## 💻 实操步骤 1：步骤一：实现 15 分钟高分答辩结构校验器与评委打分模型

- **目标文件**：`versions/day65/src/day65_defense_planner.py`
- **老陈代码旁白**：校验答辩 PPT 的时间配比是否符合高分结构，并提供企业评委高频追问库与加权评分计算器。


In [ ]:
"""
Day 65 - 模块一：15 分钟毕业答辩结构校验器与企业评委评分模型
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List


@dataclass
class DefenseSection:
    name: str
    minutes: int
    is_core_tech: bool


@dataclass
class JudgeScoreCard:
    architecture: float   # 0~100 (权重 30%)
    live_demo: float      # 0~100 (权重 25%)
    metrics_cost: float   # 0~100 (权重 25%)
    qa_defense: float     # 0~100 (权重 20%)

    def weighted_total(self) -> float:
        total = (
            self.architecture * 0.30
            + self.live_demo * 0.25
            + self.metrics_cost * 0.25
            + self.qa_defense * 0.20
        )
        return round(total, 2)

    def grade(self) -> str:
        s = self.weighted_total()
        if s >= 90:
            return "优秀 (S)"
        if s >= 80:
            return "良好 (A)"
        if s >= 70:
            return "合格 (B)"
        return "需复辩 (C)"


def validate_defense_plan(sections: List[DefenseSection], target_minutes: int = 15) -> Dict[str, object]:
    total = sum(s.minutes for s in sections)
    core_minutes = sum(s.minutes for s in sections if s.is_core_tech)
    core_ratio = round(core_minutes / max(1, total), 4)
    issues: List[str] = []
    if total != target_minutes:
        issues.append(f"总时长应为 {target_minutes} 分钟，当前为 {total} 分钟")
    if core_ratio < 0.65:
        issues.append(f"核心技术与演示占比 {core_ratio:.0%} 低于 65% 红线，背景铺垫过长")
    return {"valid": len(issues) == 0, "total_minutes": total, "core_ratio": core_ratio, "issues": issues}


JUDGE_QA_BANK = [
    {
        "topic": "RAG 幻觉控制",
        "question": "如何证明你们的 RAG 不会胡编乱造财务制度？",
        "key_points": ["BM25+向量+RRF混合召回", "Rerank低分置信度拒答门禁", "强制[DOC-xx]引用溯源", "Ragas Faithfulness 量化报告"],
    },
    {
        "topic": "Agent 安全边界",
        "question": "如果大模型抽风把 5 万元报销单直接提交了怎么办？",
        "key_points": ["确定性代码层风险策略拦截", "超2000元挂起PENDING_APPROVAL", "HITL主管审批恢复与终态幂等校验"],
    },
    {
        "topic": "成本与并发",
        "question": "全公司几千人用，Token 费用会不会爆炸？",
        "key_points": ["L1公共FAQ语义缓存", "L2本地7B与云端旗舰分级路由", "租户日配额80%预警/100%硬熔断"],
    },
]


if __name__ == "__main__":
    plan = [
        DefenseSection("业务痛点与PRD目标", 2, False),
        DefenseSection("核心架构与技术选型", 4, True),
        DefenseSection("端到端现场Demo演示", 4, True),
        DefenseSection("Ragas评测、压测与降本", 3, True),
        DefenseSection("踩坑复盘与演进路线", 2, False),
    ]
    print("答辩节奏校验:", validate_defense_plan(plan))
    score = JudgeScoreCard(architecture=92, live_demo=95, metrics_cost=90, qa_defense=88)
    print("模拟评委得分:", score.weighted_total(), score.grade())


### 🔍 步骤 1 代码深度解析

把答辩节奏规则化、评分维度透明化，帮助学员在彩排前一眼看出'是不是背景讲太长、干货太短'。

**预期运行输出参考：**
```text
答辩节奏校验: {'valid': True, 'total_minutes': 15, 'core_ratio': 0.7333, 'issues': []}
模拟评委得分: 91.6 优秀 (S)
```


## 💻 实操步骤 2：步骤二：实现防翻车现场 Demo 编排与降级验收器

- **目标文件**：`versions/day65/src/day65_live_demo_runner.py`
- **老陈代码旁白**：依次执行 3 个高光演示场景，实时服务可用时走实时调用，异常时自动切至确定性兜底快照并记录模式，确保台上零翻车。


In [ ]:
"""
Day 65 - 模块二：防翻车现场 Demo 编排器（实时主链路 + 确定性降级快照）
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, List


@dataclass
class DemoStep:
    scene_name: str
    live_fn: Callable[[], str]
    fallback_snapshot: str
    expected_keyword: str


@dataclass
class DemoStepOutcome:
    scene_name: str
    mode: str  # LIVE | FALLBACK
    output: str
    passed: bool


def run_demo_rehearsal(steps: List[DemoStep]) -> List[DemoStepOutcome]:
    outcomes: List[DemoStepOutcome] = []
    for st in steps:
        try:
            out = st.live_fn()
            mode = "LIVE"
        except Exception:
            out = st.fallback_snapshot
            mode = "FALLBACK"
        passed = st.expected_keyword in out
        outcomes.append(DemoStepOutcome(st.scene_name, mode, out, passed))
    return outcomes


if __name__ == "__main__":
    demo_steps = [
        DemoStep(
            scene_name="场景1：专有错误码混合检索+引用溯源",
            live_fn=lambda: "ERR_4291 表示部门日Token配额已触发硬熔断。[DOC-02]",
            fallback_snapshot="[快照] ERR_4291 表示部门日Token配额已触发硬熔断。[DOC-02]",
            expected_keyword="[DOC-02]",
        ),
        DemoStep(
            scene_name="场景2：高危报销 HITL 人工审批挂起",
            live_fn=lambda: "金额 4500 元超过上限，已进入 PENDING_APPROVAL（APR-8821）",
            fallback_snapshot="[快照] 金额 4500 元超过上限，已进入 PENDING_APPROVAL（APR-8821）",
            expected_keyword="PENDING_APPROVAL",
        ),
        DemoStep(
            scene_name="场景3：越狱攻击安全护栏拦截",
            live_fn=lambda: "BLOCKED: 检测到 Prompt 注入攻击，已拒绝执行",
            fallback_snapshot="[快照] BLOCKED: 检测到 Prompt 注入攻击，已拒绝执行",
            expected_keyword="BLOCKED",
        ),
    ]
    for r in run_demo_rehearsal(demo_steps):
        print(f"[{r.mode}] {r.scene_name} -> passed={r.passed}")


### 🔍 步骤 2 代码深度解析

每个演示场景都绑定了断言关键词与降级快照，上台前 10 分钟跑一遍彩排即可确认所有演示路径绿灯通过。

**预期运行输出参考：**
```text
[LIVE] 场景1：专有错误码混合检索+引用溯源 -> passed=True
[LIVE] 场景2：高危报销 HITL 人工审批挂起 -> passed=True
[LIVE] 场景3：越狱攻击安全护栏拦截 -> passed=True
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day65/tests/test_day65.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 65 单元测试：15 分钟答辩节奏校验与防翻车演示编排"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day65_defense_planner import DefenseSection, JudgeScoreCard, validate_defense_plan
from src.day65_live_demo_runner import DemoStep, run_demo_rehearsal


def test_defense_plan_validation():
    good = [
        DefenseSection("痛点", 2, False),
        DefenseSection("架构", 4, True),
        DefenseSection("Demo", 4, True),
        DefenseSection("指标", 3, True),
        DefenseSection("复盘", 2, False),
    ]
    bad = [
        DefenseSection("背景废话", 8, False),
        DefenseSection("架构与Demo", 7, True),
    ]
    assert validate_defense_plan(good)["valid"] is True
    assert validate_defense_plan(bad)["valid"] is False


def test_judge_score_card():
    sc = JudgeScoreCard(90, 92, 90, 88)
    assert sc.weighted_total() == 90.1
    assert sc.grade() == "优秀 (S)"


def test_demo_runner_live_and_fallback():
    def broken_api() -> str:
        raise ConnectionError("现场会场Wi-Fi断了")

    steps = [
        DemoStep("实时场景", lambda: "命中 [DOC-01]", "快照 [DOC-01]", "[DOC-01]"),
        DemoStep("断网降级场景", broken_api, "[快照] PENDING_APPROVAL", "PENDING_APPROVAL"),
    ]
    res = run_demo_rehearsal(steps)
    assert res[0].mode == "LIVE" and res[0].passed is True
    assert res[1].mode == "FALLBACK" and res[1].passed is True

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：补充 2 道评委追问的标准应答条目

**题目要求：**
编写函数 `filter_qa_by_keyword(qa_bank, keyword)`，支持按关键词快速检索答辩应答要点。

**思路提示：** 匹配 `keyword in item['topic'] or keyword in item['question']`。


In [ ]:
from src.day65_defense_planner import JUDGE_QA_BANK


def filter_qa_by_keyword(qa_bank: list[dict], keyword: str) -> list[dict]:
    return [q for q in qa_bank if keyword in q["topic"] or keyword in q["question"]]


hits = filter_qa_by_keyword(JUDGE_QA_BANK, "幻觉")
assert len(hits) == 1 and "RAG" in hits[0]["topic"]
print("追问库检索通过:", hits[0]["topic"])


**答案解析与评分标准：**

考前按主题高频刷追问要点，比死记硬背整段稿子更自然流畅。


### 作业 2（企业实战）：企业实战：计算 3 分钟电梯演讲（Elevator Pitch）字数合规校验

**题目要求：**
按正常语速 220 字/分钟估算，编写 `check_pitch_length(text, minutes=3)` 判断讲稿是否在合理字数区间（`[minutes*180, minutes*250]`）。

**思路提示：** 剔除空白字符后统计长度并判断区间。


In [ ]:
def check_pitch_length(text: str, minutes: int = 3) -> bool:
    n = len("".join(text.split()))
    return minutes * 180 <= n <= minutes * 250


sample = "星辰智脑是面向企业内部知识问答与办公自动化的多智能体平台。" * 22
assert check_pitch_length(sample, minutes=3) is True
print("3分钟电梯演讲字数校验通过")


**答案解析与评分标准：**

控制好字数区间能避免台上超时被打断或语速过快让评委听不清重点。


### 作业 3（架构进阶）：架构进阶：统计多轮彩排的演示稳定性（Live 成功率）

**题目要求：**
编写 `calc_live_stability(outcomes)` 计算实时模式（`mode == 'LIVE'` 且 `passed == True`）的占比，低于 0.95 提醒上线前修复主服务。

**思路提示：** 统计满足 `o.mode == 'LIVE' and o.passed` 的数量除以总数。


In [ ]:
from src.day65_live_demo_runner import DemoStepOutcome


def calc_live_stability(outcomes: list[DemoStepOutcome]) -> float:
    ok = sum(1 for o in outcomes if o.mode == "LIVE" and o.passed)
    return round(ok / max(1, len(outcomes)), 4)


res = calc_live_stability([
    DemoStepOutcome("s1", "LIVE", "ok", True),
    DemoStepOutcome("s2", "LIVE", "ok", True),
    DemoStepOutcome("s3", "FALLBACK", "ok", True),
])
assert res == 0.6667
print("彩排实时稳定性评估:", res)


**答案解析与评分标准：**

降级快照只是最后保险丝，彩排时仍需把主链路稳定性打磨到 100%。


---

## 🎯 老陈收尾复盘

恭喜各位顺利通过星辰智脑毕业答辩！今天的答辩逻辑、量化指标话术和防翻车演练，直接就能复用在接下来每一场大厂技术终面的'项目介绍'环节。明天咱们打磨简历和 GitHub 作品集！
